# PyCAM-SIMA Dask checkpoint fan-out

This standalone Notebook demonstrates the task-oriented execution mode. It does not start `NotebookSession` and does not keep an MPI job alive behind a socket. Dask first submits one 24-rank base PBS/MPI segment, retains its immutable Python-owned checkpoint as a Future, and then starts three independent 24-rank branch segments from that common state.

## 1. Execution model

```text
Jupyter + local Dask Client
           │
           ├── base PBS job: 24 MPI ranks × 10 steps
           │                    │
           │                    └── immutable checkpoint Future
           │                                  │
           ├──────────────────────────────────┼── control:    5 steps
           ├──────────────────────────────────┼── no-kessler: 5 steps
           └──────────────────────────────────┴── warm:        5 steps
```

The base process exits after writing its checkpoint. Each branch restores private NumPy arrays and creates a new `MPI.COMM_WORLD`; this is checkpoint/restart fan-out, not operating-system `fork()`.

## 2. Configure one experiment

Run this cell again to create a fresh timestamp before repeating the experiment. Existing branch directories are deliberately never overwritten.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from netCDF4 import Dataset
from pycam_sima import BranchSpec, DaskExperimentClient, FieldEdit

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/dask_notebook_trials' / f'fanout-{stamp}'
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'branches'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

print('pycam_sima', pycam_sima.__version__)
print('experiment root:', experiment_root)

## 3. Create the Dask controller

The three local Dask workers only orchestrate PBS submissions. Every actual model segment still runs on 24 MPI ranks.

In [ ]:
if 'client' in globals():
    client.close()

client = Client(
    processes=False,
    n_workers=3,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
)
client

## 4. Submit the common base and three branches

Executing this cell submits four PBS jobs in total: one base job followed by three branch jobs. `summaries()` waits until all branch jobs finish.

In [ ]:
base = experiments.submit_base(
    BranchSpec('base', steps=10)
)

branches = experiments.fork(
    base,
    (
        BranchSpec('control', steps=5),
        BranchSpec(
            'no-kessler',
            steps=5,
            disable_schemes=('kessler',),
        ),
        BranchSpec(
            'warm',
            steps=5,
            field_edits=(
                FieldEdit('air_temperature', 'add', 1.0),
            ),
        ),
    ),
)

summaries = experiments.summaries(branches)
summaries

## 5. What the summary tells you

Each branch should report `parent_branch='base'`, final `step=15`, 15 history samples, its PBS job ID, run/history/checkpoint/log paths, and serialized checkpoint size. This metadata is small; it does not download the full checkpoint Future into the Notebook process.

In [ ]:
[
    {
        'branch': name,
        'step': summary['step'],
        'history_samples': summary['history_samples'],
        'pbs_job_id': summary['pbs_job_id'],
        'checkpoint_GiB': summary['snapshot_nbytes'] / 1024**3,
        'history_dir': summary['history_dir'],
        'log_path': summary['log_path'],
    }
    for name, summary in summaries.items()
]

## 6. Inspect any final Python-owned field

Every branch checkpoint contains the complete 214-field StatePool for all 24 ranks. The following helper reads one rank-local NumPy array without starting MPI.

In [ ]:
def final_field(branch, field, rank=0):
    checkpoint_file = (
        Path(summaries[branch]['checkpoint_dir'])
        / f'rank-{rank:03d}.npz'
    )
    with np.load(checkpoint_file, allow_pickle=False) as arrays:
        return arrays[field].copy()

control_temperature = final_field('control', 'air_temperature')
warm_temperature = final_field('warm', 'air_temperature')
temperature_difference = warm_temperature - control_temperature

{
    'rank': 0,
    'shape': control_temperature.shape,
    'control_mean': float(control_temperature.mean()),
    'warm_mean': float(warm_temperature.mean()),
    'maximum_absolute_difference': float(np.abs(temperature_difference).max()),
    'bitwise_identical': bool(np.array_equal(control_temperature, warm_temperature)),
}

## 7. Inspect a global field after every model step

Each branch history directory inherits the 10 base timestamps and adds 5 branch timestamps. These NetCDF files contain the 26 configured global diagnostics. Unlike the final rank-local checkpoint, this gives one global field value at every completed model step.

In [ ]:
def history_statistics(branch, variable):
    summary = summaries[branch]
    history_dir = summary.get(
        'history_dir',
        Path(summary['checkpoint_dir']).parent / 'history',
    )
    files = sorted(Path(history_dir).glob('*.nc'))
    records = []
    for path in files:
        with Dataset(path) as dataset:
            values = np.asarray(dataset[variable][0])
            records.append({
                'step': int(dataset['nsteph'][0]),
                'file': path.name,
                'minimum': float(values.min()),
                'maximum': float(values.max()),
                'mean': float(values.mean()),
            })
    return records

control_rain_by_step = history_statistics('control', 'RAINQM')
no_kessler_rain_by_step = history_statistics('no-kessler', 'RAINQM')

{
    'control_last': control_rain_by_step[-1],
    'no_kessler_last': no_kessler_rain_by_step[-1],
    'control_all_steps': control_rain_by_step,
}

## 8. Observation boundary

This Dask mode observes all 214 StatePool fields at segment boundaries and the 26 NetCDF diagnostics after every step. It does not pause a running branch inside a step. To branch or inspect all fields at every step, submit chained one-step segments (`steps=1`) so that each task boundary produces another complete checkpoint.

In [ ]:
client.close()
print('Dask client closed; PBS results remain under', experiment_root)